# 02 — Classical ML Baseline (Heart Murmur Detection)

This notebook implements the tabular baseline for the **CirCor DigiScope** murmur classification task.
It follows the official project protocol:
- 5-Fold Patient-Level Cross-Validation on the development set (`folds 0..4`);
- Held-out test set (`fold -1`) kept strictly separate for final evaluation;
- Official PhysioNet 2022 Weighted Accuracy metric (`Present: 5`, `Unknown: 3`, `Absent: 1`).

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features_tabular import get_patient_dataset
from src.metrics import CLASSES, weighted_accuracy, report, confusion_matrix
from src.baseline import evaluate_model_cv

## 1. Acoustic Feature Dataset

Summary statistics (mean & std across recording locations) of:
- MFCCs (13 coefficients + deltas)
- Zero-Crossing Rate (ZCR)
- RMS Energy
- Spectral Centroid & Spectral Rolloff

In [2]:
df = get_patient_dataset()
print(f'Total patients: {len(df)}')
print('Folds distribution:')
print(pd.crosstab(df['fold'], df['murmur'], margins=True))
df.head(3)

Total patients: 942
Folds distribution:
murmur  Absent  Present  Unknown  All
fold                                 
-1         139       36       14  189
0          111       29       11  151
1          112       28       11  151
2          111       28       11  150
3          111       29       11  151
4          111       29       10  150
All        695      179       68  942


,patient_id,murmur,group,fold,duration_patient_mean,zcr_mean_patient_mean,zcr_std_patient_mean,rms_mean_patient_mean,rms_std_patient_mean,sc_mean_patient_mean,...,mfcc_delta_10_std_patient_std,mfcc_11_mean_patient_std,mfcc_11_std_patient_std,mfcc_delta_11_mean_patient_std,mfcc_delta_11_std_patient_std,mfcc_12_mean_patient_std,mfcc_12_std_patient_std,mfcc_delta_12_mean_patient_std,mfcc_delta_12_std_patient_std,num_recordings
0,2530,Absent,2530,-1,19.584,0.139052,0.040034,0.042497,0.030580,366.006077,...,0.126776,4.101125,0.435473,0.085052,0.134542,2.935308,0.375945,0.033421,0.084236,4
1,9979,Present,9979,3,20.616,0.095476,0.024940,0.109102,0.023481,326.719782,...,0.130567,1.988320,0.902009,0.011503,0.164179,1.124670,0.909041,0.042418,0.225712,4
2,9983,Unknown,9983,0,22.364,0.133121,0.040053,0.034869,0.017304,411.702399,...,0.248991,2.256527,0.519932,0.043418,0.138931,2.377202,0.523630,0.015988,0.133614,4


## 2. Model 1: Random Forest Baseline (5-Fold CV)

In [3]:
rep_rf = evaluate_model_cv('rf')


  Evaluating Baseline: RF (5-Fold Patient-Level CV)
  Dev set: 753 patients, 123 features

  Fold 0: Weighted Accuracy = 0.5848
  Fold 1: Weighted Accuracy = 0.5684
  Fold 2: Weighted Accuracy = 0.5211
  Fold 3: Weighted Accuracy = 0.5156
  Fold 4: Weighted Accuracy = 0.5804

--- Cross-Validation Summary ---
Mean Fold WA: 0.5541 +/- 0.0297

Overall Out-of-Fold Weighted Accuracy (Standard Argmax): 0.5541
Patient-Level Confusion Matrix (rows = predicted, cols = true):
         Present  Unknown  Absent
Present       52        4      43
Unknown        5       16      27
Absent        86       34     486
Per-class recall: {'Present': np.float64(0.364), 'Unknown': np.float64(0.296), 'Absent': np.float64(0.874)}

--- Clinical Cost-Sensitive Decision Adjustment ---
Weighted Accuracy with Class Weight Utility: 0.5024
Confusion Matrix with Utility Weighting:
         Present  Unknown  Absent
Present      130       32     492
Unknown       12       22      60
Absent         1        0       4
Pe

## 3. Model 2: HistGradientBoosting Baseline (5-Fold CV)

In [4]:
rep_hgb = evaluate_model_cv('hgb')


  Evaluating Baseline: HGB (5-Fold Patient-Level CV)
  Dev set: 753 patients, 123 features

  Fold 0: Weighted Accuracy = 0.5398
  Fold 1: Weighted Accuracy = 0.5333
  Fold 2: Weighted Accuracy = 0.5106
  Fold 3: Weighted Accuracy = 0.5606
  Fold 4: Weighted Accuracy = 0.5490

--- Cross-Validation Summary ---
Mean Fold WA: 0.5386 +/- 0.0168

Overall Out-of-Fold Weighted Accuracy (Standard Argmax): 0.5387
Patient-Level Confusion Matrix (rows = predicted, cols = true):
         Present  Unknown  Absent
Present       44        3      29
Unknown        2       12      11
Absent        97       39     516
Per-class recall: {'Present': np.float64(0.308), 'Unknown': np.float64(0.222), 'Absent': np.float64(0.928)}

--- Clinical Cost-Sensitive Decision Adjustment ---
Weighted Accuracy with Class Weight Utility: 0.5604
Confusion Matrix with Utility Weighting:
         Present  Unknown  Absent
Present       57        5      64
Unknown        2       15      19
Absent        84       34     473
P

## 4. Model 3: Logistic Regression Baseline (5-Fold CV)

In [5]:
rep_lr = evaluate_model_cv('lr')


  Evaluating Baseline: LR (5-Fold Patient-Level CV)
  Dev set: 753 patients, 123 features

  Fold 0: Weighted Accuracy = 0.6125
  Fold 1: Weighted Accuracy = 0.5684
  Fold 2: Weighted Accuracy = 0.5563
  Fold 3: Weighted Accuracy = 0.5675
  Fold 4: Weighted Accuracy = 0.5629

--- Cross-Validation Summary ---
Mean Fold WA: 0.5735 +/- 0.0199

Overall Out-of-Fold Weighted Accuracy (Standard Argmax): 0.5736
Patient-Level Confusion Matrix (rows = predicted, cols = true):
         Present  Unknown  Absent
Present       78        9     130
Unknown       10       20      54
Absent        55       25     372
Per-class recall: {'Present': np.float64(0.545), 'Unknown': np.float64(0.37), 'Absent': np.float64(0.669)}

--- Clinical Cost-Sensitive Decision Adjustment ---
Weighted Accuracy with Class Weight Utility: 0.5701
Confusion Matrix with Utility Weighting:
         Present  Unknown  Absent
Present      101       14     242
Unknown       11       21      65
Absent        31       19     249
Per